# What changes when a water-index threshold changes?

Distinguish a spectral index from a water classification.

**Task** · Compare MNDWI areas at thresholds 0 and 0.2 near the Han River with long-term water occurrence.

Run from top to bottom. Predict the result before changing code, then compare it with the output.

## Before you start

Use a CPU runtime. You need a registered Earth Engine Cloud project and permission to use it. See the [setup guide](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md). Authenticate with your own account before querying data.

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Your Cloud project ID registered for Earth Engine
if not PROJECT_ID.strip():
    raise ValueError("Enter your registered Google Cloud project ID in PROJECT_ID")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
def mask_s2(image):
    scl = image.select('SCL')
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return (image.select(['B2', 'B3', 'B4', 'B8', 'B11'])
            .multiply(0.0001).updateMask(clear)
            .copyProperties(image, ['system:time_start']))
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi).filterDate('2020-05-01', '2020-10-01')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
              .map(mask_s2))

image = collection.median().clip(roi)
mndwi = image.expression('(g-s)/(g+s)', {'g':image.select('B3'),'s':image.select('B11')}).rename('MNDWI')
ndvi = image.expression('(n-r)/(n+r)', {'n':image.select('B8'),'r':image.select('B4')}).rename('NDVI')
water0 = mndwi.gt(0).rename('water0')
water02 = mndwi.gt(0.2).rename('water02')
areas = ee.Image.pixelArea().multiply(water0.addBands(water02)).divide(1e6).reduceRegion(ee.Reducer.sum(),roi,20,maxPixels=5_000_000)
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(roi)

In [ ]:
print('threshold areas (km2):', areas.getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi,10)
Map.addLayer(image, {'bands':['B4','B3','B2'],'min':0,'max':0.3}, 'RGB')
Map.addLayer(mndwi, {'min':-1,'max':1,'palette':['brown','white','blue']}, 'MNDWI')
Map.addLayer(water0.selfMask(), {'palette':['cyan']}, 'MNDWI > 0')
Map.addLayer(water02.selfMask(), {'palette':['blue']}, 'MNDWI > 0.2')
Map.addLayer(jrc, {'min':0,'max':100,'palette':['white','blue']}, 'JRC occurrence 1984–2021 (%)')
Map

Explain how increasing the threshold changes the candidate water area. Use the RGB image to find possible confusion with clouds, shadows and built-up surfaces.

**Scope** · JRC occurrence is the fraction of valid observations classified as water over 1984–2021. It is not field truth for the 2020 composite. Differences between the maps do not all indicate classification errors or flooding. B11 is originally a 20 m band, so this activity uses a 20 m calculation scale.

## Data and interpretation

- [Sentinel-2 SR](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)

- [JRC Global Surface Water 1.4](https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater)

Observation years define fixed examples, independent of the teaching year. The code demonstrates processing methods; inspect actual maps and values after authentication.